# 数据预处理

## 输入
- 要求：CSV 文件包含 `timestamp` 和 `load_kwh`（累计电能）列
- 建筑原始数据位置：`data/buildings/{建筑名}.CSV`

## 输出
- `data/buildings/{建筑名}_预处理后.csv`：负荷数据预处理结果
- `data/buildings/{建筑名}_特征.csv`：负荷 + 天气 + 时间特征（用于模型训练）
- `data/天气_预处理后.csv`：天气数据预处理结果（共享）

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from typing import Tuple
import joblib

# 从 src 模块导入预处理函数和共享配置
from src import (
    build_hr_load_series,
    fill_missing,
    detect_outliers,
    encode_cyclical_feature,
    BASE_DIR,
    BUILDINGS_DIR,
    SCALER_DIR,
    BUILDINGS,
    TIME_COL,
    TARGET_COL,
)

# 天气和日历数据路径
WEATHER_PATH = BASE_DIR / '天气.CSV'
WEATHER_OUTPUT_PATH = BASE_DIR / '天气_预处理后.csv'
CALENDAR_PATH = BASE_DIR / '2024日历.csv'

# 构建建筑数据配置列表
BUILDING_CONFIGS = [
    {
        'name': name,
        'load_path': BUILDINGS_DIR / f'{name}.CSV',
    }
    for name in BUILDINGS
]

# 连续天气特征列表
WEATHER_CONTINUOUS_COLS = [
    '温度(℃)', '风力(级)', '风速(km/h)',
    '气压(hPa)', '湿度(%)', '能见度(km)', '云量%', '降水量对数变换'
]

In [2]:
def preprocess_load_data(
    input_path: Path,
    output_path: Path,
    cumulative_col: str = "load_kwh",
    hr_load_col: str = "hourly_kwh",
) -> pd.DataFrame:
    raw_df = pd.read_csv(input_path, encoding='utf-8')
    clean_df = build_hr_load_series(raw_df, TIME_COL, cumulative_col, hr_load_col)
    clean_df["is_outlier_mad"] = detect_outliers(clean_df[hr_load_col])
    s_no_outlier = clean_df[hr_load_col].mask(clean_df["is_outlier_mad"], np.nan)
    clean_df[TARGET_COL] = fill_missing(s_no_outlier).round(2)
    clean_df.reset_index().to_csv(output_path, index=False, encoding='utf-8-sig')
    return clean_df


def process_weather_data(
    input_path: Path,
    output_path: Path,
) -> Tuple[pd.DataFrame, StandardScaler]:
    """
    处理天气数据并进行标准化（所有建筑使用同一份标准化后的天气数据）
    """
    weather_df = pd.read_csv(input_path, encoding='gbk')
    
    if '时间' in weather_df.columns:
        weather_df = weather_df.rename(columns={'时间': TIME_COL})
    weather_df[TIME_COL] = pd.to_datetime(weather_df[TIME_COL], errors='coerce')
    weather_df = weather_df.dropna(axis=0, how='any').copy()
    
    precip_col = '降水量(mm)'
    if precip_col in weather_df.columns:
        weather_df['是否降水'] = weather_df[precip_col] > 0
        weather_df['降水量对数变换'] = np.log1p(weather_df[precip_col])
        weather_df = weather_df.drop(columns=[precip_col])
    else:
        weather_df['是否降水'] = False
        weather_df['降水量对数变换'] = 0.0
    
    cat_cols = weather_df.select_dtypes(exclude=['number']).columns.tolist()
    cat_cols = [c for c in cat_cols if c not in [TIME_COL, '是否降水']]

    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded_arr = ohe.fit_transform(weather_df[cat_cols])
    encoded_cols = ohe.get_feature_names_out(cat_cols)
    encoded_df = pd.DataFrame(encoded_arr, columns=encoded_cols, index=weather_df.index)
    encoded_df = encoded_df.astype('bool')
    weather_df = pd.concat([weather_df.drop(columns=cat_cols), encoded_df], axis=1)
    
    weather_df = weather_df.sort_values(TIME_COL).reset_index(drop=True)
    
    # 标准化连续天气特征（用全量天气数据计算 scaler）
    weather_cols = [c for c in WEATHER_CONTINUOUS_COLS if c in weather_df.columns]
    weather_scaler = StandardScaler()
    weather_df[weather_cols] = weather_scaler.fit_transform(weather_df[weather_cols])

    joblib.dump(weather_scaler, SCALER_DIR / 'weather_scaler.joblib')
    weather_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    return weather_df, weather_scaler


def create_time_features(
    timestamps: pd.Series,
    calendar_df: pd.DataFrame,
) -> pd.DataFrame:
    time_features_df = pd.DataFrame({TIME_COL: timestamps})
    time_features_df['date'] = time_features_df[TIME_COL].dt.normalize()
    time_features_df['hour'] = time_features_df[TIME_COL].dt.hour
    time_features_df = pd.merge(time_features_df, calendar_df, on='date', how='left')
    
    time_features_df['hour_sin'], time_features_df['hour_cos'] = encode_cyclical_feature(
        time_features_df['hour'], 24
    )
    time_features_df['day_of_week_sin'], time_features_df['day_of_week_cos'] = encode_cyclical_feature(
        time_features_df['day_of_week'] - 1, 7
    )
    time_features_df['month_sin'], time_features_df['month_cos'] = encode_cyclical_feature(
        time_features_df['month'] - 1, 12
    )
    time_features_df['is_holiday'] = time_features_df['is_holiday'].astype('bool')
    time_features_df = time_features_df.drop(columns=['date', 'hour', 'day_of_week', 'month'])
    
    return time_features_df


def merge_features(
    load_df: pd.DataFrame,
    weather_df: pd.DataFrame,
    time_features_df: pd.DataFrame,
    output_path: Path,
) -> pd.DataFrame:
    load_cols = [TIME_COL, TARGET_COL]
    load_clean_df = load_df.reset_index()[load_cols]
    
    merged_df = pd.merge(load_clean_df, weather_df, on=TIME_COL, how='left')
    merged_df = pd.merge(merged_df, time_features_df, on=TIME_COL, how='inner')
    
    merged_df = merged_df.sort_values(TIME_COL).reset_index(drop=True)
    merged_df.to_csv(output_path, index=False, encoding='utf-8-sig', float_format='%.10f')
    
    return merged_df

## 批量处理所有建筑数据

In [3]:
# 天气数据标准化并保存 scaler
weather_df, weather_scaler = process_weather_data(WEATHER_PATH, WEATHER_OUTPUT_PATH)

# 读取日历数据
calendar_df = pd.read_csv(CALENDAR_PATH, encoding='utf-8-sig')
calendar_df['date'] = pd.to_datetime(calendar_df['date'].astype(str), format='%Y%m%d')

# 存储所有建筑的处理结果
building_data = {}

for building in BUILDING_CONFIGS:
    load_output_path = BUILDINGS_DIR / f'{building["name"]}_预处理后.csv'
    load_df = preprocess_load_data(building['load_path'], load_output_path)

    timestamps = load_df.reset_index()[TIME_COL]
    time_features_df = create_time_features(timestamps, calendar_df)

    feature_output_path = BUILDINGS_DIR / f'{building["name"]}_特征.csv'
    feature_df = merge_features(load_df, weather_df, time_features_df, feature_output_path)

    building_data[building['name']] = {
        'load_df': load_df,
        'feature_df': feature_df,
    }

## 数据汇总

In [4]:
# 汇总所有建筑的处理结果
summary_data = []
for name, data in building_data.items():
    load_df = data['load_df']
    feature_df = data['feature_df']
    
    summary_data.append({
        '建筑名称': name,
        '负荷数据形状': load_df.shape,
        '特征数据形状': feature_df.shape,
        '有效样本数': load_df[TARGET_COL].notna().sum(),
        '时间范围': f"{load_df.index.min()} ~ {load_df.index.max()}",
    })

summary_df = pd.DataFrame(summary_data)
summary_df

,建筑名称,负荷数据形状,特征数据形状,有效样本数,时间范围
0,南阶,"(8784, 4)","(8784, 46)",8783,2024-01-01 00:00:00 ~ 2024-12-31 23:00:00
1,西阶,"(8784, 4)","(8784, 46)",8552,2024-01-01 00:00:00 ~ 2024-12-31 23:00:00
2,环工,"(8784, 4)","(8784, 46)",8783,2024-01-01 00:00:00 ~ 2024-12-31 23:00:00
3,土木,"(8784, 4)","(8784, 46)",8783,2024-01-01 00:00:00 ~ 2024-12-31 23:00:00
4,粉体,"(8784, 4)","(8784, 46)",8675,2024-01-01 00:00:00 ~ 2024-12-31 23:00:00
